In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import random
import pickle
import json
import joblib
import torch
from abc import ABC, abstractmethod
import numpy as np
import pandas as pd
import torch.nn as nn

class BaseModel(ABC):

    QUANTILES   = [0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975]
    PRED_LENGTH = 28
    SEED        = 25
    TARGET_START = 1914
    DEBUG_ONE_CHUNK = False

    def __init__(self, data_dir="data", output_dir="outputs"):
        random.seed(self.SEED)
        np.random.seed(self.SEED)
        torch.manual_seed(self.SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.data_dir   = data_dir
        self.output_dir = output_dir
        os.makedirs(data_dir,   exist_ok=True)
        os.makedirs(output_dir, exist_ok=True)

        self.train_raw = self.val_raw = self.test_raw = None
        self.calendar  = self.prices  = self.item_weights = None
        self.train_processed = self.val_processed = self.test_processed = None
        self.model = None

    @property
    @abstractmethod
    def model_name(self) -> str: ...

    @abstractmethod
    def preprocess(self): ...
    # Input:  self.train_raw, self.val_raw, self.test_raw
    # Output: self.train_processed, self.val_processed, self.test_processed
    #         (format is model-specific: tensors, lgb.Dataset, DataFrames, etc.)

    @abstractmethod
    def train(self): ...
    # Input:  self.train_processed, self.val_processed
    # Must:
    #   1. Set self.model
    #   2. Save weights → output_dir/{model_name}.pth (or .pkl for LGBM)

    @abstractmethod
    def predict(self): ...
    # Input: self.test_processed, self.output_dir
    # Must:
    # 1. Load model from: output_dir/{model_name}.pth/pkl
    # 2. Only use d_1886-d_1913 as context (if needed)
    # 3. Predict 9 quantiles for d_1914-d_1941 in preds_df: a DataFrame (30490 * 28 rows) with columns:
    #     id | day_ahead (1-28) | q0.025 | q0.05 | q0.1 | q0.25 | q0.5 | q0.75 | q0.9 | q0.95 | q0.975
    # 4. Sort predictions by id, day_ahead
    # 5. Make sure quantiles are non-decreasing and >= 0
    # 6. Save predictions → output_dir/{model_name}_predictions.csv
    # Output: preds_df

    # Shared methods

    def load_and_split_data(self):
        """
        Downloads (or loads from cache) M5 data and processes it into long-format dataframes (1 row per item and day).
        Split into train (d_1-d_1773), val (d_1774-d_1885), and test (d_1886-d_1941).
        Save raw splits to cache for loading when training models.

        Output: self.train_raw, self.val_raw, self.test_raw, self.item_weights
        """
        # Step 1: load data
        cache = os.path.join(self.data_dir, "raw_split.pkl")

        if os.path.exists(cache):
            with open(cache, "rb") as f:
                d = pickle.load(f)
            print("Loaded cached data splits.")

        else:
            base     = "https://huggingface.co/datasets/kashif/M5/resolve/main"
            sales    = pd.read_csv(f"{base}/sales_train_evaluation.csv")
            calendar = pd.read_csv(f"{base}/calendar.csv")
            prices   = pd.read_csv(f"{base}/sell_prices.csv")
            print("Downloaded M5 data.")

            # Step 2: melt sales to long format
            id_cols  = [c for c in sales.columns if not c.startswith("d_")]
            day_cols = [c for c in sales.columns if c.startswith("d_")]
            sales_long = sales[id_cols + day_cols].melt(
                id_vars=id_cols, var_name='d', value_name='sales'
            )
            sales_long['d_num'] = sales_long['d'].str[2:].astype(int)

            # Step 3: merge calendar data and set dtypes
            for col in ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']:
                calendar[col] = calendar[col].fillna('none').astype('category')
            calendar['weekday'] = calendar['weekday'].astype('category')
            calendar['date']    = pd.to_datetime(calendar['date'])
            for col in ['snap_CA', 'snap_TX', 'snap_WI', 'wday', 'month']:
                calendar[col] = calendar[col].astype('int8')
            calendar['year'] = calendar['year'].astype('int16')  # fix year encoding to handle years 2011-2016
            sales_long = sales_long.merge(calendar, on='d', how='left')

            # Step 4: merge daily prices
            sales_long = sales_long.merge(
                prices[['store_id', 'item_id', 'wm_yr_wk', 'sell_price']],
                on=['store_id', 'item_id', 'wm_yr_wk'], how='left'
            )

            # Step 5: add is_available flag and forward-fill missing prices
            sales_long['is_available'] = sales_long['sell_price'].notna().astype('int8')
            sales_long = sales_long.sort_values(['id', 'd_num']).reset_index(drop=True)
            sales_long['sell_price'] = (
                sales_long.groupby('id')['sell_price']
                .transform(lambda x: x.ffill().fillna(0.0))
            )

            # Step 6: check missing values
            missing = sales_long.isnull().sum()
            missing = missing[missing > 0]
            assert len(missing) == 0, f"Missing values found:\n{missing}"

            # Step 7: split into train/val/test
            total_days = len(day_cols)
            train_end  = total_days - 6 * self.PRED_LENGTH   # 1773
            val_end    = total_days - 2 * self.PRED_LENGTH   # 1885

            train_raw = sales_long[sales_long['d_num'] <= train_end].reset_index(drop=True)
            val_raw   = sales_long[(sales_long['d_num'] > train_end) & (sales_long['d_num'] <= val_end)].reset_index(drop=True)
            test_raw  = sales_long[sales_long['d_num'] > val_end].reset_index(drop=True)

            assert train_raw['d_num'].nunique() == 1773
            assert val_raw['d_num'].nunique()   == 112
            assert test_raw['d_num'].nunique()  == 56
            print(f"Train: d_1-d_{train_end} | Val: d_{train_end+1}-d_{val_end} | Test: d_{val_end+1}-d_{total_days}")

            # Step 8: calculate revenue weights on last 28 training days only
            last28 = train_raw[train_raw['d_num'] > train_end - 28]
            train_rev = (last28['sales'] * last28['sell_price']).groupby(last28['id']).sum()
            item_weights = (train_rev / train_rev.sum()).rename('weight')

            # Step 9: save data splits to cache
            d = dict(train_raw=train_raw, val_raw=val_raw, test_raw=test_raw,
                        item_weights=item_weights)

            # save all splits and weights into raw_split.pkl
            with open(cache, "wb") as f:
                pickle.dump(d, f)
            print(f"Cached to {cache}")

        self.train_raw    = d["train_raw"]
        self.val_raw      = d["val_raw"]
        self.test_raw     = d["test_raw"]
        self.item_weights = d["item_weights"]

        return self.train_raw, self.val_raw, self.test_raw, self.item_weights

    # Evaluation

    def _build_pinball_tensor(self, preds_df: pd.DataFrame):
        """
        Shared setup for WSPL and CRPS. Requires self.train_raw, self.test_raw.
        Returns y_mat (N,28), q_arr (N,9,28), ids, scale.
        """
        q_cols  = [f"q{q}" for q in self.QUANTILES]

        # Extract test targets for d_1914-d_1941 (28 days)
        test_targets = (
            self.test_raw[self.test_raw['d_num'] >= self.TARGET_START]
            .pivot(index='id', columns='d_num', values='sales')
            .sort_index()
        )
        ids = test_targets.index.tolist()
        y_mat = test_targets.values.astype(np.float32)  # Shape: (N_series, 28)

        # Reshape predictions to match
        preds_pivot = (
            preds_df.set_index(['id', 'day_ahead'])[q_cols]
            .unstack('day_ahead')
            .sort_index(axis=1)
            .loc[ids]
)
        q_arr = preds_pivot.values.reshape(len(ids), len(self.QUANTILES), self.PRED_LENGTH)

        train_s = self.train_raw.sort_values(['id', 'd_num'])

        prev = train_s.groupby('id')['sales'].shift(1)
        mask = prev.notna()
        abs_diff = (train_s.loc[mask, 'sales'] - prev[mask]).abs()

        scale = abs_diff.groupby(train_s.loc[mask, 'id']).mean()

        scale = scale.clip(lower=1e-6)

        # Checks for debugging
        assert y_mat.shape == (len(ids), 28), f"y_mat shape mismatch: {y_mat.shape}"
        assert q_arr.shape == (len(ids), 9, 28), f"q_arr shape mismatch: {q_arr.shape}"

        return y_mat, q_arr, ids, scale

    def compute_wspl(self, y_mat, q_arr, ids, scale) -> float:
        """Weighted scaled pinball loss across 9 quantiles and 28 forecast days."""
        q_vals  = np.array(self.QUANTILES)
        errors  = y_mat[:, None, :] - q_arr
        pinball = np.maximum(q_vals[None, :, None] * errors,
                             (q_vals[None, :, None] - 1) * errors)
        wspl_per_series = pinball.mean(axis=(1, 2)) / scale.reindex(ids).values
        weights         = self.item_weights.reindex(ids).fillna(0).values
        return float(np.dot(weights, wspl_per_series))

    def compute_crps(self, y_mat, q_arr, ids) -> float:
        """Weighted CRPS approximated as 2x mean pinball loss across quantiles and forecast days."""
        q_vals  = np.array(self.QUANTILES)
        errors  = y_mat[:, None, :] - q_arr
        pinball = np.maximum(q_vals[None, :, None] * errors,
                            (q_vals[None, :, None] - 1) * errors)
        crps_per_series = 2 * pinball.mean(axis=(1, 2))
        weights = self.item_weights.reindex(ids).fillna(0).values
        return float(np.dot(weights, crps_per_series))

    def compute_coverage(self, preds_df: pd.DataFrame, y_mat, ids) -> dict:
        """
        Coverage error (actual - nominal) for 4 prediction intervals.
        Positive = over-coverage, negative = under-coverage.
        """
        intervals = {
            0.50: ("q0.25",  "q0.75"),
            0.80: ("q0.1",   "q0.9"),
            0.90: ("q0.05",  "q0.95"),
            0.95: ("q0.025", "q0.975"),
        }
        preds_indexed = (
            preds_df.set_index(['id', 'day_ahead'])
            .reindex(pd.MultiIndex.from_product(
                [ids, range(1, self.PRED_LENGTH + 1)],
                names=['id', 'day_ahead']))
        )
        coverage_errors = {}
        for nominal, (lower_col, upper_col) in intervals.items():
            lower   = preds_indexed[lower_col].values.reshape(len(ids), self.PRED_LENGTH)
            upper   = preds_indexed[upper_col].values.reshape(len(ids), self.PRED_LENGTH)
            covered = ((y_mat >= lower) & (y_mat <= upper)).mean()
            coverage_errors[f"coverage_error_{int(nominal*100)}pct"] = round(
                float(covered - nominal), 6)
        return coverage_errors

    def _validate_preds(self, preds_df):
        """
        Validate predictions for easier debugging of inference pipeline.
        """
        expected_ids = self.test_raw['id'].unique()
        q_cols = [f"q{q}" for q in self.QUANTILES]
        assert set(expected_ids).issubset(set(preds_df['id'])), "Missing IDs"
        assert set(preds_df['day_ahead'].unique()) == set(range(1, 29)), "day_ahead must be 1-28"
        assert all(c in preds_df.columns for c in q_cols), "Missing quantile columns"
        assert (preds_df[q_cols].diff(axis=1).iloc[:, 1:] >= 0).all().all(), "Quantiles non-monotonic"
        assert (preds_df[q_cols] >= 0).all().all(), "Negative quantile predictions"

    def evaluate(self, preds_df: pd.DataFrame = None) -> dict:
        """
        Compute WSPL, CRPS and coverage error from a predictions DataFrame.
        Saves results to output_dir/{model_name}_results.json.
        Requires: raw_split.pkl in data_dir. Run load_and_split_data() first.
        """
        if preds_df is None:
          path = f"{self.output_dir}/{self.model_name}_predictions.csv"
          preds_df = pd.read_csv(path)

        self._validate_preds(preds_df)
        # Step 1: load data and weights
        cache = os.path.join(self.data_dir, "raw_split.pkl")
        assert os.path.exists(cache), "Run load_and_split_data() first."
        with open(cache, "rb") as f:
            d = pickle.load(f)
        self.train_raw    = d["train_raw"]
        self.test_raw     = d["test_raw"]
        self.item_weights = d["item_weights"]

        # Step 2: build pinball tensor
        y_mat, q_arr, ids, scale = self._build_pinball_tensor(preds_df)

        # Step 3: compute evaluation metrics
        results = {
            "model" : self.model_name,
            "wspl"  : round(self.compute_wspl(y_mat, q_arr, ids, scale), 6),
            "crps"  : round(self.compute_crps(y_mat, q_arr, ids), 6),
            **self.compute_coverage(preds_df, y_mat, ids),
        }

        # Step 4: save results to json
        out_path = os.path.join(self.output_dir, f"{self.model_name}_results.json")
        with open(out_path, "w") as f:
            json.dump(results, f, indent=2)
        return results

    # Pipelines

    def run_training_pipeline(self):
    # Combines loading and splitting data, preprocessing, and training into a single pipeline
        self.load_and_split_data()
        print("Finished shared data processing.")
        self.preprocess()
        print("Finished model-specific data processing.")
        self.train()
        print("Finished model training.")

    def run_inference_pipeline(self):
    # Combines inference and evaluation into a single pipeline
        self.load_and_split_data()  # needs train_raw and test_raw, if training pipeline was not run before this
        preds_df = self.predict()
        print("Finished model inference.")
        results = self.evaluate(preds_df)
        print("Finished model evaluation.")
        return results


In [3]:
class FeatureNN(nn.Module):
    def __init__(self, n_items, input_dim):
        super().__init__()
        self.emb = nn.Embedding(n_items, 8)

        self.backbone = nn.Sequential(
            nn.Linear(input_dim + 8, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        # predicts a SINGLE value instead of predicting quantiles which is what LightGBM does
        # but even then we don't really use this output, only the representations it has used along the way
        # the NN does NOT forecast, just learns good representations which is why we are using MSE loss here instead of pinball loss
        self.head = nn.Linear(32, 1)

    def forward(self, x, item):
        x = torch.cat([x, self.emb(item)], dim=1)
        features = self.backbone(x)
        out = self.head(features)
        return out.squeeze(-1), features


In [4]:
class LightGBM_NN(BaseModel):

    @property
    def model_name(self):
        return "LightGBM_NN_Hybrid"

    def preprocess(self):
        from pandas.api.types import is_numeric_dtype
        print("starting preprocessing")
        # LightGBM cannot pick up time-series features easily so we must create these features
        # The NN will also use these features to help understand the time series patterns
        def add_features(df):
                df = df.sort_values(["id","d_num"])
                grp = df.groupby("id")["sales"]
                # creating previous sales at t-1, t-7, t-14, t-28 days
                # to learn past sale demand
                for lag in [1,7,14,28]:
                    df[f"lag_{lag}"] = grp.shift(lag)
                # average sales over last 7 and 28 days
                for w in [7,28]:
                    df[f"roll_mean_{w}"] = grp.shift(1).rolling(w).mean()
                df["roll_std_7"] = grp.shift(1).rolling(7).std()
                # to model long term growth
                df["trend"] = df["d_num"].astype(np.int16)
                # to help with seasonality
                df["week_of_year"] = df["wm_yr_wk"].astype(np.int16)
                # transforms day of week into a circle to visualise where we are in the week better
                # this will help then to predict sales
                df["dow_sin"] = np.sin(2*np.pi*df["wday"]/7).astype(np.float32)
                df["dow_cos"] = np.cos(2*np.pi*df["wday"]/7).astype(np.float32)

                return df
        # add these features into the datasets
        train = add_features(self.train_raw)
        val = add_features(self.val_raw)
        test = add_features(self.test_raw)

        # looping over the datasets to apply the same preprocessing
        for dataset in [train, val, test]:
            # get the IDs / categorical variables
            cat_cols = ["item_id","dept_id","cat_id","store_id","state_id"]
            # define the numerical features (everything except whats not in cat_cols)
            num_cols = [c for c in dataset.columns if c not in cat_cols]
            # loop through numerical columns
            for col in num_cols:
                col_data = dataset[col]
                # check they are actually numerical & converts NaNs to 0
                if is_numeric_dtype(col_data):
                    col_data = col_data.mask(np.isinf(col_data), np.nan)
                    col_data = col_data.fillna(0)
                    # note we are using float 32 to reduce memory usage
                    dataset[col] = col_data.astype(np.float32)
                else:
                    dataset[col] = col_data

        self.train_processed = train
        self.val_processed = val
        self.test_processed = test

    def train(self):
          import numpy as np
          import time
          import lightgbm as lgb
          from sklearn.preprocessing import StandardScaler
          from pandas.api.types import is_numeric_dtype
          import os
          import torch
          import torch.nn as nn

          from torch.utils.data import TensorDataset, DataLoader

          self.output_dir = f"/content/drive/MyDrive/applied_deep_learning/outputs/run_{int(time.time())}"
          os.makedirs(self.output_dir, exist_ok=True)
          print("\nTRAINING STARTING\n")
          quantiles = self.QUANTILES
          models = {}
          df = self.train_processed.copy()

          # create the target
          df["target"] = df.groupby("id")["sales"].shift(-1)

          # perform log transformation to help shrink the intervals
          df["target"] = np.log1p(df["target"])
          # get rid of target column for training
          df = df.dropna(subset=["target"])
          # encode categorical variables
          cat_cols = ["item_id","dept_id","cat_id","store_id","state_id"]
          cat_maps = {}
          for c in cat_cols:
              uniques = df[c].unique()
              cat_maps[c] = {k:i for i,k in enumerate(uniques)}
              df[c] = df[c].map(cat_maps[c]).astype(np.int16)

          self.n_items = len(cat_maps["item_id"])
          # clean the numerical features
          for col in df.columns:
              if is_numeric_dtype(df[col]):
                  col_data = df[col]
                  col_data = col_data.mask(np.isinf(col_data), np.nan)
                  col_data = col_data.fillna(0)
                  df[col] = col_data.astype(np.float32)

          # get the features
          drop_cols = ["sales","d_num","time_idx","target"]
          features = [
              "lag_1","lag_7","lag_14","lag_28",
              "roll_mean_7","roll_mean_28", "roll_std_7",
              "trend","week_of_year",
              "dow_sin","dow_cos",
              "wday","month","sell_price","is_available"
          ]

          self.features = features
          # scale the features
          scaler = StandardScaler()
          X = df[features].values.astype(np.float32)
          X_scaled = scaler.fit_transform(X)
          self.scaler = scaler
          # create tensors
          X_t = torch.tensor(X_scaled, dtype=torch.float32)
          item_t = torch.tensor(df["item_id"].values, dtype=torch.long)
          y_t = torch.tensor(df["target"].values.astype(np.float32))
          dataset = TensorDataset(X_t, item_t, y_t)
          # train in batches to help reduce memory issue
          loader = DataLoader(
              dataset,
              batch_size=8192,
              shuffle=True
          )

          model_nn = FeatureNN(self.n_items, len(features)).to(self.device)
          opt = torch.optim.Adam(model_nn.parameters(), lr=0.0002)

          print("Training NN")
          # 5 epochs is enough as one takes roughly 10-15 mins
          for epoch in range(5):
              total_loss = 0
              for xb, itemb, yb in loader:
                  xb = xb.to(self.device)
                  itemb = itemb.to(self.device)
                  yb = yb.to(self.device)

                  pred, emb = model_nn(xb, itemb)
                  loss = nn.MSELoss()(pred, yb)

                  opt.zero_grad()
                  loss.backward()
                  opt.step()

                  total_loss += loss.item()

              print(f"Epoch {epoch+1} | Loss: {total_loss:.4f}")


          model_nn.eval()
          emb_list = []

          with torch.no_grad():
              for xb, itemb, _ in loader:
                  xb = xb.to(self.device)
                  itemb = itemb.to(self.device)

                  _, emb = model_nn(xb, itemb)
                  emb_list.append(emb.cpu().numpy())

          emb = np.vstack(emb_list)

          # combine scaled features and embeddings
          # NN is noisy so use 0.4*
          X_aug = np.concatenate((X_scaled, 0.4*emb), axis=1)

          y = df["target"].values.astype(np.float32)
          # train different light gbm for each quantile
          for q in quantiles:
              print(f"Training LightGBM q={q}")

              dtrain = lgb.Dataset(X_aug, y)

              model = lgb.train(
                  {
                      "objective": "quantile",
                      "alpha": q,
                      "learning_rate": 0.03,
                      "num_leaves": 64,
                      "verbose": -1
                  },
                  dtrain,
                  num_boost_round=200
              )
              # save model
              model.save_model(f"{self.output_dir}/lgb_q_{q}.txt")
              models[q] = model
              print("Finished training")

          import joblib

          joblib.dump(scaler, f"{self.output_dir}/scaler.pkl")
          joblib.dump(features, f"{self.output_dir}/features.pkl")
          joblib.dump(cat_maps, f"{self.output_dir}/cat_maps.pkl")
          joblib.dump(self.n_items, f"{self.output_dir}/n_items.pkl")
          torch.save(model_nn.state_dict(), f"{self.output_dir}/nn_embedding.pth")

          self.models = models

          print("\nTRAINING COMPLETE\n")


    def predict(self):
        import numpy as np
        import pandas as pd
        import torch
        import lightgbm as lgb
        import joblib
        import os
        # add features to test set
        def add_predict_features(buf, last, h):
            dow = int((last["wday"] + h - 1) % 7)

            return {
                "lag_1": buf[-1],
                "lag_7": buf[-7],
                "lag_14": buf[-14],
                "lag_28": buf[-28],

                "roll_mean_7": np.mean(buf[-7:]),
                "roll_mean_28": np.mean(buf),
                "roll_std_7": np.std(buf[-7:], ddof=1),

                "trend": last["d_num"] + h,
                 "week_of_year" : (last["wm_yr_wk"] + (h // 7) - 1) % 52 + 1,

                "dow_sin": np.sin(2*np.pi*dow/7),
                "dow_cos": np.cos(2*np.pi*dow/7),

                "wday": dow,
                "month": last["month"],
                "sell_price": last["sell_price"],
                "is_available": last["is_available"],
            }

        print("Loading models...")

        quantiles = self.QUANTILES
        # load models
        models = {
            q: lgb.Booster(model_file=f"{self.output_dir}/lgb_q_{q}.txt")
            for q in quantiles
        }

        scaler = joblib.load(f"{self.output_dir}/scaler.pkl")
        features = joblib.load(f"{self.output_dir}/features.pkl")
        cat_maps = joblib.load(f"{self.output_dir}/cat_maps.pkl")
        n_items = joblib.load(f"{self.output_dir}/n_items.pkl")
        # load NN
        model_nn = FeatureNN(n_items, len(features))
        model_nn.load_state_dict(torch.load(f"{self.output_dir}/nn_embedding.pth"))
        model_nn.eval().to(self.device)

        # prepare test data
        df = self.test_processed.copy()
        df = df[(df["d_num"] >= 1886) & (df["d_num"] <= 1913)]

        cat_cols = ["item_id","dept_id","cat_id","store_id","state_id"]
        for c in cat_cols:
            df[c] = df[c].map(cat_maps[c]).fillna(-1).astype(np.int16)

        rows = []

        print("Generating forecasts")
        for item_id, item_df in df.groupby("id"):

            item_df = item_df.sort_values("d_num")

            # last 28 sales is thebuffer
            buf = item_df["sales"].values[-28:].tolist()
            last = item_df.iloc[-1]

            item_encoded = cat_maps["item_id"].get(last["item_id"], 0)
            # start forecasting for each day
            for h in range(1, 29):
                feat_dict = add_predict_features(buf, last, h)
                x_df = pd.DataFrame([feat_dict])
                x_df = x_df[features]

                x_raw = x_df.values.astype(np.float32)
                x_scaled = scaler.transform(x_raw)
                # get the embeddings
                x_t = torch.tensor(x_scaled, dtype=torch.float32).to(self.device)
                item_t = torch.tensor([item_encoded], dtype=torch.long).to(self.device)
                with torch.no_grad():
                    _, emb = model_nn(x_t, item_t)
                    emb = emb.cpu().numpy()

                x_aug = np.concatenate((x_scaled, 0.4*emb), axis=1)

                rec = {"id": item_id, "day_ahead": h}
                preds_step = {}

                for q in quantiles:
                    p = float(models[q].predict(x_aug)[0])
                    p = np.expm1(p)
                    p = max(0.0, p)

                    preds_step[q] = p
                    rec[f"q{q}"] = p

                buf.append(preds_step[0.5])
                buf = buf[-28:]

                rows.append(rec)

        # create into a dataframe and csv file
        preds_df = pd.DataFrame(rows)
        preds_df = preds_df.sort_values(["id","day_ahead"]).reset_index(drop=True)

        q_cols = [f"q{q}" for q in quantiles]

        preds_df[q_cols] = preds_df[q_cols].clip(lower=0)
        preds_df[q_cols] = np.maximum.accumulate(preds_df[q_cols].values, axis=1)

        out = f"{self.output_dir}/{self.model_name}_predictions.csv"
        preds_df.to_csv(out, index=False)

        print(f"Saved → {out}")

        return preds_df




In [5]:
m = LightGBM_NN()
m.load_and_split_data()
m.preprocess()
m.train()
m.predict()
m.evaluate()

Downloaded M5 data.
Train: d_1-d_1773 | Val: d_1774-d_1885 | Test: d_1886-d_1941
Cached to data/raw_split.pkl
starting preprocessing

TRAINING STARTING

Training NN...
Epoch 1 | Loss: 1291.0694
Epoch 2 | Loss: 1195.1253
Epoch 3 | Loss: 1187.7706
Epoch 4 | Loss: 1182.9787
Epoch 5 | Loss: 1177.2933
Training LightGBM q=0.025
Finished training
Training LightGBM q=0.05
Finished training
Training LightGBM q=0.1
Finished training
Training LightGBM q=0.25
Finished training
Training LightGBM q=0.5
Finished training
Training LightGBM q=0.75
Finished training
Training LightGBM q=0.9
Finished training
Training LightGBM q=0.95
Finished training
Training LightGBM q=0.975
Finished training

TRAINING COMPLETE

Loading models...
Generating forecasts...
Saved → /content/drive/MyDrive/applied_deep_learning/outputs/run_1775608947/LightGBM_NN_Hybrid_predictions.csv


{'model': 'LightGBM_NN_Hybrid',
 'wspl': 0.462539,
 'crps': 1.239251,
 'coverage_error_50pct': 0.18317,
 'coverage_error_80pct': 0.050286,
 'coverage_error_90pct': 0.060096,
 'coverage_error_95pct': 0.029698}